# Sales Assessment Data Cleaning Pipeline

This notebook cleans the raw `Sales_Assessment_Data.xlsx` file by addressing **8 data quality issues**:

| # | Issue | Description |
|---|-------|-------------|
| 1 | Duplicate Shipment IDs | 21 duplicate IDs retained across lifecycle stages |
| 2 | Status field inconsistency | 35+ raw variants → 8 canonical statuses |
| 3 | Trailer Type inconsistency | 120+ variants → 15 canonical types |
| 4 | Shipment Type inconsistency | ~40 variants → 7 canonical types |
| 5 | Margin Total mismatch | Net vs Gross margin — add derived columns |
| 6 | Missing values in key fields | Categorical nulls filled with placeholders |
| 7 | Zero Buy/Sell Totals | Flag rows with $0 financials for ops review |
| 8 | TL edge case | Handled within Shipment Type standardisation |

---

## Cell 1 — Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Resolve the notebook's directory so all relative paths below
# work regardless of where the notebook is launched from.
# ─────────────────────────────────────────────────────────────
path = Path().resolve()

print("Working directory:", path)
print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

# Sanity-check: confirm input file is reachable from here
raw_file = path / 'input' / 'Sales_Assessment_Data.xlsx'
print(f"Input file found: {raw_file.exists()} → {raw_file}")

Working directory: /Users/soumya/Projects/freight-brokerage-profitability-analysis/src
pandas  version : 3.0.1
numpy   version : 2.4.3
Input file found: True → /Users/soumya/Projects/freight-brokerage-profitability-analysis/src/input/Sales_Assessment_Data.xlsx


## Cell 2 — Load Raw Data

In [2]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    """
    Load the raw Sales Assessment Excel file into a DataFrame.

    Parameters
    ----------
    input_path : Path
        Full path to the source .xlsx file.

    Returns
    -------
    pd.DataFrame
        Raw, unmodified DataFrame exactly as it comes from Excel.

    Notes
    -----
    - No transformations are applied here; this cell is purely I/O.
    - Shape is printed immediately so we can verify row/column counts
      before any cleaning step touches the data.
    """
    df = pd.read_excel(input_path)
    print(f"Raw dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
    return df


# ── Run ──────────────────────────────────────────────────────
RAW_FILE = path / 'input' / 'Sales_Assessment_Data.xlsx'
df = load_raw_data(RAW_FILE)
df.head()

Raw dataset loaded: 100,432 rows, 29 columns


,Pickup Date,Shipment ID,Customer Name,Sales Rep Name,Carrier Rep,Margin Total,Buy Total,Sell Total,Status,LSP Name,...,Destination ZIP Code,Destination Country,Customer Contact,Mileage,Total Weight,Shipment Type,Trailer Type,Linehaul Carrier Name,Account Manager,SSR
0,2023-01-02,116313930,Durable Solutions,Peyton King,Alex Patel,592.34,2612.66,2941.75,Delivered,Wilmington,...,22963.0,USA,Drew Roberts,1122.25,43603.0,Truckload,Van,STERLING CARRIERS,NaN,NaN
1,2023-01-02,116465757,Blue Ridge Holdings,"Parker Parker, Parker Parker",Parker Parker,190.54,682.43,904.11,Delivered,Charlotte,...,84990.0,USA,Alex Anderson,489.05,8000.0,Truckload,24 ft Straight Truck,FALCON FREIGHT LINES,NaN,NaN
2,2023-01-03,116457156,Redwood Distribution,"Parker Parker, Sam Williams",Sam Williams,121.44,1036.70,1123.78,Delivered,Charlotte,...,59350.0,USA,Sage James,383.26,25282.0,Truckload,32 ft Hotshot,ZEPHYR TRUCKING LLC,Parker Parker,Parker Parker
3,2023-01-03,116464754,Landmark Solutions,NaN,Morgan Evans,211.50,447.94,684.48,Delivered,Lake Zurich,...,13595.0,USA,Reese Mitchell,171.38,2769.0,LTL,Van,FORGEPOINT CARRIERS,Alex Hall,Devon Lee
4,2023-01-03,116415186,Evergreen Corp,"Morgan Roberts, Alex Chen",Alex Chen,89.90,1744.77,1803.40,Delivered,Charlotte,...,27743.0,USA,Casey Reed,591.00,44669.0,Truckload,FLATBED,DELTA FREIGHT LINES,Val Lewis,Val Lewis


## Cell 3 — Status Priority Lookup & Normalisation Helper

Used **only during deduplication** (Issue 1). Not the same as the final status standardisation (Issue 2).

In [3]:
# ─────────────────────────────────────────────────────────────
# STATUS_PRIORITY maps each canonical status to an integer
# rank so we can sort duplicates and keep the most-advanced
# lifecycle stage.
#
# Convention: LOWER number = HIGHER priority (kept first after sort).
#   1  Delivered     — shipment physically received by consignee
#   2  Complete      — all admin closed out
#   3  In Transit    — en-route
#   4  Out for Delivery — local delivery leg started
#   5  Dispatched    — driver assigned, not yet moving
#   6  Committed     — carrier committed but not yet dispatched
#   7  Quote         — pre-booking stage
#   8  Canceled      — voided; kept last so it's only retained if
#                      no active status exists for that Shipment ID
# ─────────────────────────────────────────────────────────────
STATUS_PRIORITY = {
    'Delivered':        1,
    'Complete':         2,
    'In Transit':       3,
    'Out for Delivery': 4,
    'Dispatched':       5,
    'Committed':        6,
    'Quote':            7,
    'Canceled':         8,
}


def normalize_status_for_priority(s) -> str:
    """
    Map any raw Status string to a canonical group name so it can
    be looked up in STATUS_PRIORITY for deduplication sorting.

    This is intentionally separate from `standardize_status` (Cell 5)
    because here we only need to identify the priority bucket;
    we don't need a fully cleaned label yet.

    Parameters
    ----------
    s : any
        Raw Status value from the DataFrame (may be NaN or a string).

    Returns
    -------
    str
        One of the 8 canonical group names used as keys in STATUS_PRIORITY.
        Defaults to 'Quote' for NaN or unrecognised values so they sort
        toward the bottom (priority 7) rather than getting a missing-key
        error from the dict lookup.

    Examples
    --------
    >>> normalize_status_for_priority('dlvrd')
    'Delivered'
    >>> normalize_status_for_priority('Cancled')   # typo variant
    'Canceled'
    >>> normalize_status_for_priority(None)
    'Quote'
    """
    # Treat NaN / None as lowest-priority pre-shipment status
    if pd.isna(s):
        return 'Quote'

    s_lower = str(s).strip().lower()

    # ── Delivered ─────────────────────────────────────────────
    # Covers typos ('deliverd') and common abbreviation ('dlvrd')
    if s_lower in ['delivered', 'deliverd', 'deliver', 'dlvrd']:
        return 'Delivered'

    # ── Complete ──────────────────────────────────────────────
    elif s_lower in ['complete', 'completed']:
        return 'Complete'

    # ── In Transit ────────────────────────────────────────────
    elif s_lower in ['in transit', 'in-transit', 'intransit']:
        return 'In Transit'

    # ── Out for Delivery ──────────────────────────────────────
    elif s_lower in ['out for delivery', 'ofd']:
        return 'Out for Delivery'

    # ── Dispatched ────────────────────────────────────────────
    elif s_lower == 'dispatched':
        return 'Dispatched'

    # ── Committed ─────────────────────────────────────────────
    elif s_lower == 'committed':
        return 'Committed'

    # ── Quote (pre-booking variants) ──────────────────────────
    # 'ready' and 'sent' are synonyms used in some source systems
    elif s_lower in ['quote', 'quoted', 'ready', 'sent']:
        return 'Quote'

    # ── Canceled ──────────────────────────────────────────────
    elif s_lower in ['canceled', 'cancelled', 'cncld', 'cancled']:
        return 'Canceled'

    # ── Default fallback ──────────────────────────────────────
    # Unrecognised values are treated as pre-shipment / lowest priority
    return 'Quote'


# Quick smoke-test
test_cases = ['dlvrd', 'Cancled', 'OFD', 'in-transit', None, 'SENT']
for tc in test_cases:
    print(f"{str(tc):<20} → {normalize_status_for_priority(tc)}")

dlvrd                → Delivered
Cancled              → Canceled
OFD                  → Out for Delivery
in-transit           → In Transit
None                 → Quote
SENT                 → Quote


## Cell 4 — Issue 1: Remove Duplicate Shipment IDs

In [4]:
def deduplicate_by_shipment_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resolve duplicate Shipment IDs by retaining the single row
    that represents the most advanced lifecycle stage.

    Background
    ----------
    21 Shipment IDs appear on multiple rows, sometimes with different
    dates and statuses (e.g., 'Quote' AND 'Delivered' for the same ID).
    This is a lifecycle artefact: the same shipment was inserted into
    the source system at multiple stages rather than updated in-place.

    Strategy
    --------
    1. Assign each row a numeric priority score based on its Status
       (lower score = higher priority; see STATUS_PRIORITY dict).
    2. Sort by [Shipment ID ↑, priority score ↑, Pickup Date ↓] so
       the most advanced AND most recent row floats to the top.
    3. Drop all but the first row per Shipment ID.
    4. Remove the temporary priority column.

    Parameters
    ----------
    df : pd.DataFrame
        Raw or partially cleaned DataFrame containing a 'Status' column
        and a 'Shipment ID' column.

    Returns
    -------
    pd.DataFrame
        DataFrame with one row per Shipment ID; duplicates removed.
    """
    # Step 1 — Compute numeric priority for each row's Status.
    # normalize_status_for_priority() maps messy raw values to canonical
    # group names; STATUS_PRIORITY converts those to sort-friendly integers.
    # Fallback to 9 for any group name not in the dict (shouldn't happen).
    df['_status_priority'] = df['Status'].apply(
        lambda s: STATUS_PRIORITY.get(normalize_status_for_priority(s), 9)
    )

    # Step 2 — Sort so the row we want to KEEP rises to position 0
    # within each Shipment ID group:
    #   - _status_priority ASC  → highest lifecycle stage first
    #   - Pickup Date      DESC → if same priority, most recent date first
    df.sort_values(
        ['Shipment ID', '_status_priority', 'Pickup Date'],
        ascending=[True, True, False],
        inplace=True
    )

    # Step 3 — Drop all duplicate rows, keeping only the first
    # (= highest-priority, most-recent) row per Shipment ID
    before = len(df)
    df = df.drop_duplicates(subset='Shipment ID', keep='first')
    after = len(df)

    print(f"Issue 1 — Duplicates removed: {before - after} rows dropped "
          f"({before:,} → {after:,})")

    # Step 4 — Drop the temporary helper column; it has served its purpose
    df.drop(columns=['_status_priority'], inplace=True)

    return df


# ── Run ──────────────────────────────────────────────────────
df = deduplicate_by_shipment_id(df)
print(f"Rows after dedup: {len(df):,}")

Issue 1 — Duplicates removed: 20 rows dropped (100,432 → 100,412)
Rows after dedup: 100,412


## Cell 5 — Issue 2: Standardise Status Field

In [5]:
def standardize_status(s) -> str:
    """
    Map 35+ raw Status variants to 8 canonical status labels.

    Background
    ----------
    The source system contains 35+ unique Status strings that represent
    only 8 real states. Problems observed:
      - Mixed case        : 'delivered', 'DELIVERED', 'Delivered'
      - Typos             : 'Deliverd', 'Cancled'
      - Abbreviations     : 'dlvrd', 'OFD', 'Cncld'
      - Synonyms          : 'Completed' ≡ 'Complete', 'Quoted' ≡ 'Quote'

    Canonical Output Values
    -----------------------
    Delivered | Complete | In Transit | Out for Delivery
    Dispatched | Committed | Quote | Canceled | Unknown

    Parameters
    ----------
    s : any
        Raw Status value (string or NaN).

    Returns
    -------
    str
        One of the 9 canonical labels above.
        Returns 'Unknown' for NaN or unrecognised strings.

    Examples
    --------
    >>> standardize_status('dlvrd')
    'Delivered'
    >>> standardize_status('Cancled')
    'Canceled'
    >>> standardize_status(None)
    'Unknown'
    """
    # NaN / None → Unknown (we cannot infer the shipment state)
    if pd.isna(s):
        return 'Unknown'

    s_lower = str(s).strip().lower()

    # ── Delivered ─────────────────────────────────────────────
    if s_lower in ['delivered', 'deliverd', 'deliver', 'dlvrd']:
        return 'Delivered'

    # ── Complete ──────────────────────────────────────────────
    # Treated as a distinct terminal state from 'Delivered'
    # (some carriers mark admin closure separately)
    elif s_lower in ['complete', 'completed']:
        return 'Complete'

    # ── In Transit ────────────────────────────────────────────
    elif s_lower in ['in transit', 'in-transit', 'intransit', 'in_transit']:
        return 'In Transit'

    # ── Out for Delivery ──────────────────────────────────────
    # Final delivery leg; sub-state of In Transit but kept separate
    # because it has operational significance (driver on road now)
    elif s_lower in ['out for delivery', 'ofd']:
        return 'Out for Delivery'

    # ── Dispatched ────────────────────────────────────────────
    elif s_lower == 'dispatched':
        return 'Dispatched'

    # ── Committed ─────────────────────────────────────────────
    elif s_lower == 'committed':
        return 'Committed'

    # ── Quote ─────────────────────────────────────────────────
    # All pre-booking synonyms map here
    elif s_lower in ['quote', 'quoted', 'ready', 'sent']:
        return 'Quote'

    # ── Canceled ──────────────────────────────────────────────
    # Covers British/American spelling variants and abbreviations
    elif s_lower in ['canceled', 'cancelled', 'cncld', 'cancled']:
        return 'Canceled'

    # ── Fallback ──────────────────────────────────────────────
    else:
        return 'Unknown'


def apply_status_standardization(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply standardize_status() to the entire Status column and
    print a before/after summary.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Same DataFrame with the Status column cleaned in-place.
    """
    df['Status'] = df['Status'].apply(standardize_status)

    print(f"Issue 2 — Status standardised. "
          f"Unique values now: {df['Status'].nunique()}")
    print(df['Status'].value_counts().to_string())
    return df


# ── Run ──────────────────────────────────────────────────────
df = apply_status_standardization(df)

Issue 2 — Status standardised. Unique values now: 8
Status
Delivered           78584
Quote               16967
Canceled             2929
Complete             1242
In Transit            559
Committed              69
Out for Delivery       45
Dispatched             17


## Cell 6 — Issue 3: Standardise Trailer Type

In [6]:
def standardize_trailer(s) -> str:
    """
    Map 120+ raw Trailer Type variants to 15 canonical equipment types
    using keyword detection (case-insensitive substring matching).

    Background
    ----------
    The source data contains 120+ unique Trailer Type strings for what
    are ~12–15 real equipment categories. Problems observed:
      - Mixed case        : 'Van', 'VAN', 'van', ' Van'
      - Abbreviations     : 'P/O' for Power Only, 'OFD'
      - Verbose variants  : 'FiftyThreeReefer' for '53 ft Reefer'
      - Descriptive noise : 'van (dry)', 'Van | Dry'
      - 2,415 NULLs
      - Placeholder text  : 'Not Specified', 'Unspecified'

    Canonical Output Values
    -----------------------
    Van | Flatbed | Reefer | LTL | Power Only | Hotshot
    Straight Truck | Sprinter | Step Deck | Conestoga
    Intermodal | Air Freight | Low Boy / RGN | Tanker | Unknown

    Design Notes
    ------------
    - 'Power Only' is checked BEFORE 'Van' to prevent 'power van'
      from accidentally matching the Van branch.
    - 'Reefer' is checked BEFORE 'Van' for the same reason ('van reefer').
    - Order matters: more specific terms come first.

    Parameters
    ----------
    s : any
        Raw Trailer Type value (string or NaN).

    Returns
    -------
    str
        One of the 15 canonical labels above.
    """
    # NULL / NaN → Unknown (equipment type genuinely missing)
    if pd.isna(s):
        return 'Unknown'

    s_clean = str(s).strip().lower()

    # ── Reject non-informative placeholders ───────────────────
    if s_clean in ['not specified', 'unspecified',
                   'other - see accessorials', 'n/a', '']:
        return 'Unknown'

    # ── Power Only ────────────────────────────────────────────
    # Must appear BEFORE 'van' check to catch 'power van'-style strings
    if any(k in s_clean for k in ['power only', 'poweronly',
                                   'power-only', 'p/o']):
        return 'Power Only'

    # ── Reefer (refrigerated trailer) ─────────────────────────
    # Must appear BEFORE 'van' check ('van reefer' → Reefer, not Van)
    if any(k in s_clean for k in ['reefer', 'refrigerated', 'rf ']):
        return 'Reefer'

    # ── Flatbed ───────────────────────────────────────────────
    # Check before generic 'flat' substring to avoid partial matches
    if any(k in s_clean for k in ['flatbed', 'flat bed',
                                   'flat-bed', 'flatb']):
        return 'Flatbed'

    # ── Conestoga (rolling-tarp flatbed) ──────────────────────
    if 'conestoga' in s_clean or 'curtain side' in s_clean:
        return 'Conestoga'

    # ── Step Deck (lowered deck for taller freight) ───────────
    if any(k in s_clean for k in ['step deck', 'stepdeck']):
        return 'Step Deck'

    # ── Hotshot (small flatbed, typically < 40 ft) ────────────
    if any(k in s_clean for k in ['hotshot', 'hot shot',
                                   'hot-shot', 'hotsht']):
        return 'Hotshot'

    # ── Straight Truck / Box Truck ────────────────────────────
    # Common city-delivery vehicles; size suffixes like '26 ft' are
    # a reliable signal for straight trucks in this dataset
    if any(k in s_clean for k in ['straight truck', 'straighttruck',
                                   'straight-truck', 'city truck',
                                   '12 ft', '24 ft', '26 ft']):
        return 'Straight Truck'

    # ── Sprinter Van (smaller than straight truck) ────────────
    if 'sprinter' in s_clean:
        return 'Sprinter'

    # ── Intermodal / Container ────────────────────────────────
    if any(k in s_clean for k in ['intermodal', 'container',
                                   'ocean cont']):
        return 'Intermodal'

    # ── LTL (Less-Than-Truckload) ─────────────────────────────
    if any(k in s_clean for k in ['ltl', 'less than',
                                   'less-than', 'l.t.l.']):
        return 'LTL'

    # ── Van (dry van) — broadest fallback ─────────────────────
    # Placed AFTER all specialised checks to avoid over-matching
    if any(k in s_clean for k in ['van', 'dry van', 'dryvan']):
        return 'Van'

    # ── Air Freight ───────────────────────────────────────────
    if any(k in s_clean for k in ['air freight', 'dom. air', 'int. air']):
        return 'Air Freight'

    # ── Low Boy / RGN (heavy / oversized equipment) ───────────
    if any(k in s_clean for k in ['low boy', 'lowboy',
                                   'rgn', 'double drop']):
        return 'Low Boy / RGN'

    # ── Tanker ────────────────────────────────────────────────
    if 'tanker' in s_clean or 'liquid bulk' in s_clean:
        return 'Tanker'

    # ── Fallback ──────────────────────────────────────────────
    return 'Unknown'


def apply_trailer_standardization(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply standardize_trailer() column-wide and print summary.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    df['Trailer Type'] = df['Trailer Type'].apply(standardize_trailer)

    print(f"Issue 3 — Trailer Type standardised. "
          f"Unique values now: {df['Trailer Type'].nunique()}")
    print(df['Trailer Type'].value_counts().to_string())
    return df


# ── Run ──────────────────────────────────────────────────────
df = apply_trailer_standardization(df)

Issue 3 — Trailer Type standardised. Unique values now: 15
Trailer Type
Van               56847
Flatbed           17753
LTL                9483
Reefer             3712
Unknown            3571
Power Only         3503
Hotshot            2321
Straight Truck     1968
Sprinter            753
Step Deck           167
Intermodal          146
Conestoga           141
Tanker               25
Low Boy / RGN        12
Air Freight          10


## Cell 7 — Issue 4: Standardise Shipment Type

In [7]:
def standardize_shipment_type(s) -> str:
    """
    Normalise ~40 raw Shipment Type variants to 7 canonical labels.

    Background
    ----------
    Shipment Type describes the service model (how the freight moves),
    not the equipment. ~40 variants exist for 7 core service types.
    Issues include:
      - Abbreviations : 'TL', 'T/L', 'LTL', 'L.T.L.'
      - Spacing       : 'Truck Load', 'Truck-Load', 'Truckload'
      - Edge case     : bare 'TL' must match Truckload but
                        must NOT match 'LTL' (substring overlap risk)

    Canonical Output Values
    -----------------------
    Truckload | LTL | Drayage | Hot Shot | Intermodal
    Air Freight | Other | Unknown

    Parameters
    ----------
    s : any
        Raw Shipment Type value.

    Returns
    -------
    str
        One of the 8 canonical labels above.

    Notes
    -----
    The bare 'tl' case is handled explicitly (exact-match guard) to
    avoid matching the 'tl' substring inside 'ltl'.
    """
    if pd.isna(s):
        return 'Unknown'

    s_clean = str(s).strip().lower()

    # ── Reject placeholders ───────────────────────────────────
    if s_clean in ['unspecified', 'not specified', 'n/a', '']:
        return 'Unknown'

    # ── Truckload ─────────────────────────────────────────────
    # Note: ' tl' and 'tl ' include a space to prevent matching
    # the 'tl' that appears inside 'ltl'.
    if any(k in s_clean for k in ['truckload', 'truck load',
                                   'truck-load', 't/l', ' tl', 'tl ']):
        return 'Truckload'

    # ── Exact match for bare 'tl' ────────────────────────────
    # Separate guard because the keyword scan above only catches
    # 'tl' when surrounded by spaces; a standalone cell 'TL' needs
    # its own exact-match path.
    if s_clean == 'tl':
        return 'Truckload'

    # ── LTL (Less-Than-Truckload) ─────────────────────────────
    # Checked AFTER Truckload to avoid 'ltl' being caught by the
    # 'tl' substring rule above.
    elif any(k in s_clean for k in ['ltl', 'less than',
                                     'less-than', 'l.t.l.']):
        return 'LTL'

    # ── Drayage (port / rail-ramp short haul) ────────────────
    elif any(k in s_clean for k in ['drayage', 'dray']):
        return 'Drayage'

    # ── Hot Shot (expedited small loads) ─────────────────────
    elif any(k in s_clean for k in ['hot shot', 'hotshot', 'hot-shot']):
        return 'Hot Shot'

    # ── Intermodal ────────────────────────────────────────────
    elif 'intermodal' in s_clean:
        return 'Intermodal'

    # ── Air Freight ───────────────────────────────────────────
    elif any(k in s_clean for k in ['air freight', 'air',
                                     'dom. air', 'int. air']):
        return 'Air Freight'

    # ── Other (niche / non-standard service types) ────────────
    elif any(k in s_clean for k in ['partial', 'rf tl', 'rf ltl',
                                     'rail', 'bulk', 'services']):
        return 'Other'

    return 'Unknown'


def apply_shipment_type_standardization(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply standardize_shipment_type() column-wide and print summary.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    df['Shipment Type'] = df['Shipment Type'].apply(standardize_shipment_type)

    print(f"Issue 4 — Shipment Type standardised. "
          f"Unique values: {df['Shipment Type'].nunique()}")
    print(df['Shipment Type'].value_counts().to_string())
    return df


# ── Run ──────────────────────────────────────────────────────
df = apply_shipment_type_standardization(df)

Issue 4 — Shipment Type standardised. Unique values: 8
Shipment Type
Truckload      57214
LTL            39987
Drayage         1664
Hot Shot         946
Unknown          530
Intermodal        33
Other             27
Air Freight       11


## Cell 8 — Issue 5: Margin Consistency & Derived Columns

In [8]:
def add_margin_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resolve the Margin Total inconsistency and add three derived
    financial columns for downstream analytics.

    Background
    ----------
    For 97,849 rows, `Margin Total ≠ Sell Total - Buy Total`.
    Investigation suggests that Margin Total represents the
    **net** (adjusted) margin after accessorials, fuel surcharges,
    and other fees, while `Sell Total - Buy Total` is the
    **gross** margin before those adjustments.

    Decision
    --------
    - Preserve `Margin Total` as-is (it is the authoritative net figure).
    - Add `Gross Margin` = Sell Total - Buy Total  (pre-adjustment reference).
    - Add `Margin Pct`   = Net Margin / Sell Total × 100  (rate, for sorting).
    - Add `Negative Margin Flag` = True when Margin Total < 0  (ops alert).

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: 'Sell Total', 'Buy Total', 'Margin Total'.

    Returns
    -------
    pd.DataFrame
        Original DataFrame plus three new columns:
        'Gross Margin', 'Margin Pct', 'Negative Margin Flag'.
    """
    # ── Gross Margin (pre-accessorial) ────────────────────────
    # Simple arithmetic; rounded to 2 dp to match currency precision
    df['Gross Margin'] = (df['Sell Total'] - df['Buy Total']).round(2)

    # ── Margin % (net) ────────────────────────────────────────
    # Use np.where to guard against division by zero when
    # Sell Total = 0 (those rows are already flagged in Issue 7).
    # Result is in percentage points (e.g., 18.5 means 18.5%).
    df['Margin Pct'] = np.where(
        df['Sell Total'] != 0,
        (df['Margin Total'] / df['Sell Total'] * 100).round(2),
        np.nan   # undefined margin % when revenue is $0
    )

    # ── Negative Margin Flag ──────────────────────────────────
    # Boolean column; True = brokerage LOST money on this load.
    # Used as a critical filter in the executive dashboard.
    df['Negative Margin Flag'] = df['Margin Total'] < 0

    neg_count = df['Negative Margin Flag'].sum()
    print(f"Issue 5 — Margin columns added.")
    print(f"  Negative margin rows : {neg_count:,}")
    print(f"  Gross Margin range   : "
          f"${df['Gross Margin'].min():,.2f} – "
          f"${df['Gross Margin'].max():,.2f}")
    print(f"  Margin Pct range     : "
          f"{df['Margin Pct'].min():.1f}% – "
          f"{df['Margin Pct'].max():.1f}%")

    return df


# ── Run ──────────────────────────────────────────────────────
df = add_margin_columns(df)

Issue 5 — Margin columns added.
  Negative margin rows : 2,917
  Gross Margin range   : $-5,294.05 – $19,332.91
  Margin Pct range     : -1989.5% – 110.3%


## Cell 9 — Issue 6: Fill Missing Categorical Values

In [9]:
def fill_missing_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace NULL values in categorical columns with meaningful
    placeholder strings rather than dropping rows or imputing values.

    Background
    ----------
    Several columns have significant null volumes:
      Sales Rep Name          : 32,412 nulls
      Carrier Rep             : 53,313 nulls
      Account Manager / SSR   : ~3,418 nulls each
      Linehaul Carrier Name   : ~7,371 nulls
      Origin / Dest Country   : 14,000–28,000 nulls
      Address street fields   : variable nulls
      Destination Company     : variable nulls

    Why placeholders, not imputation?
    ----------------------------------
    - We cannot infer WHO a sales rep or carrier is from other columns.
    - Placeholder strings ('Unassigned', 'Unknown') are operationally
      meaningful and allow dashboards to segment the unassigned cohort.
    - Numerical nulls (ZIP codes, mileage, weight) are intentionally
      LEFT as NaN — filling those would introduce false precision.

    Placeholder conventions
    -----------------------
    'Unassigned'    — for people / rep fields (implies actionable gap)
    'Unknown'       — for reference/lookup fields (location, carrier)
    'Not Provided'  — for free-text address fields (display only, not joins)

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        DataFrame with categorical nulls filled; all numerical nulls
        remain unchanged.
    """
    # ── People / rep fields → 'Unassigned' ───────────────────
    # These represent missing accountability; 'Unassigned' signals
    # to ops leadership that these loads have no assigned owner.
    rep_fields = ['Sales Rep Name', 'Carrier Rep',
                  'Account Manager', 'SSR']
    for col in rep_fields:
        if col in df.columns:
            null_count = df[col].isna().sum()
            df[col] = df[col].fillna('Unassigned')
            print(f"  {col:<30} {null_count:>7,} nulls → 'Unassigned'")

    # ── Reference / lookup fields → 'Unknown' ────────────────
    # These are dimension values used in GROUP BY / filtering;
    # 'Unknown' is a valid dimension member in the dashboard.
    ref_fields = {
        'Linehaul Carrier Name': 'Unknown Carrier',
        'Origin Country':        'Unknown',
        'Destination Country':   'Unknown',
        'Destination Company Name': 'Unknown',
    }
    for col, placeholder in ref_fields.items():
        if col in df.columns:
            null_count = df[col].isna().sum()
            df[col] = df[col].fillna(placeholder)
            print(f"  {col:<30} {null_count:>7,} nulls → '{placeholder}'")

    # ── Address fields → 'Not Provided' ──────────────────────
    # Street addresses are display-only; they are never used as
    # join keys so 'Not Provided' is safe and user-friendly.
    addr_fields = ['Origin Street Address', 'Destination Street Address']
    for col in addr_fields:
        if col in df.columns:
            null_count = df[col].isna().sum()
            df[col] = df[col].fillna('Not Provided')
            print(f"  {col:<30} {null_count:>7,} nulls → 'Not Provided'")

    print("\nIssue 6 — Categorical nulls filled with placeholders.")
    return df


# ── Run ──────────────────────────────────────────────────────
print("Column-level null fills:")
df = fill_missing_categoricals(df)

Column-level null fills:
  Sales Rep Name                  32,398 nulls → 'Unassigned'
  Carrier Rep                     53,294 nulls → 'Unassigned'
  Account Manager                  3,418 nulls → 'Unassigned'
  SSR                              3,418 nulls → 'Unassigned'
  Linehaul Carrier Name            7,370 nulls → 'Unknown Carrier'
  Origin Country                  14,402 nulls → 'Unknown'
  Destination Country             14,402 nulls → 'Unknown'
  Destination Company Name        28,515 nulls → 'Unknown'
  Origin Street Address           28,005 nulls → 'Not Provided'
  Destination Street Address      28,547 nulls → 'Not Provided'

Issue 6 — Categorical nulls filled with placeholders.


## Cell 10 — Issue 7: Flag Zero Financial Values

In [10]:
def flag_zero_financials(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag rows where Buy Total or Sell Total is $0.

    Background
    ----------
    4,117 rows have $0 Buy Total; 2,805 rows have $0 Sell Total.
    Both are operationally invalid for a completed brokerage shipment —
    a freight broker always has cost (Buy) and revenue (Sell).

    Possible root causes:
      a) Quote-stage records that never progressed (no pricing yet)
      b) Data entry errors (missing values recorded as 0)
      c) Accessorial-only invoices billed separately

    Decision
    --------
    Do NOT drop these rows — they may represent legitimate Quote records
    or require manual review by operations. Instead, add a boolean flag
    column so analysts and dashboards can filter them as needed.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain 'Buy Total' and 'Sell Total' columns.

    Returns
    -------
    pd.DataFrame
        Same DataFrame with a new boolean column 'Zero_Financial_Flag'.
        True  = at least one of Buy Total / Sell Total is $0.
        False = both Buy Total and Sell Total are non-zero.
    """
    # Flag is True if EITHER Buy Total OR Sell Total equals zero.
    # The bitwise OR (|) operates element-wise on boolean Series.
    df['Zero_Financial_Flag'] = (
        (df['Buy Total'] == 0) | (df['Sell Total'] == 0)
    )

    zero_buy  = (df['Buy Total']  == 0).sum()
    zero_sell = (df['Sell Total'] == 0).sum()
    both_zero = ((df['Buy Total'] == 0) & (df['Sell Total'] == 0)).sum()
    flagged   = df['Zero_Financial_Flag'].sum()

    print(f"Issue 7 — Zero financial value flags added.")
    print(f"  $0 Buy Total  : {zero_buy:,} rows")
    print(f"  $0 Sell Total : {zero_sell:,} rows")
    print(f"  Both $0       : {both_zero:,} rows")
    print(f"  Total flagged : {flagged:,} rows (union)")

    return df


# ── Run ──────────────────────────────────────────────────────
df = flag_zero_financials(df)

Issue 7 — Zero financial value flags added.
  $0 Buy Total  : 4,117 rows
  $0 Sell Total : 2,805 rows
  Both $0       : 2,542 rows
  Total flagged : 4,380 rows (union)


## Cell 11 — Final Sort, Reset Index & Save

In [11]:
def finalise_and_save(df: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    """
    Sort the cleaned DataFrame by Pickup Date, reset the index,
    and export to Excel.

    Parameters
    ----------
    df : pd.DataFrame
        Fully cleaned DataFrame.
    output_path : Path
        Full path for the output .xlsx file.

    Returns
    -------
    pd.DataFrame
        Sorted, index-reset DataFrame (also written to disk).

    Notes
    -----
    - Sorting by Pickup Date makes downstream time-series analysis
      and Excel scroll-through easier.
    - index=False prevents pandas from writing the integer index
      as an extra column in the output file.
    """
    # ── Sort chronologically ───────────────────────────────────
    # Ascending Pickup Date ensures the earliest shipments appear
    # at the top, which is the natural expectation for time-series
    # review in Excel or BI tools.
    df.sort_values('Pickup Date', inplace=True)

    # ── Reset integer index ────────────────────────────────────
    # After dedup + sort the index is fragmented; reset to 0-based
    # sequential integers for clean slicing downstream.
    df.reset_index(drop=True, inplace=True)

    # ── Export to Excel ────────────────────────────────────────
    # Ensure the output directory exists (no-op if already present)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(output_path, index=False)

    print("=" * 60)
    print("CLEANING COMPLETE")
    print(f"Final row count    : {len(df):,}")
    print(f"Final column count : {df.shape[1]}")
    print(f"Saved to           : {output_path}")
    print("=" * 60)

    return df


# ── Run ──────────────────────────────────────────────────────
OUTPUT_FILE = path / 'output' / 'Freight_Cleaned_Dataset.xlsx'
df = finalise_and_save(df, OUTPUT_FILE)

CLEANING COMPLETE
Final row count    : 100,412
Final column count : 33
Saved to           : /Users/soumya/Projects/freight-brokerage-profitability-analysis/src/output/Freight_Cleaned_Dataset.xlsx


## Cell 12 — Summary Statistics

In [12]:
def print_summary_statistics(df: pd.DataFrame) -> None:
    """
    Print distribution summaries for the three key categorical
    columns cleaned in this pipeline, plus a financial overview.

    This serves as a final sanity-check: any unexpected canonical
    value or distribution anomaly should be investigated before
    the file is handed off to the BI / dashboard team.

    Parameters
    ----------
    df : pd.DataFrame
        Finalised cleaned DataFrame.

    Returns
    -------
    None — prints to stdout only.
    """
    separator = "─" * 50

    # ── Status distribution ────────────────────────────────────
    print("\n=== FINAL STATUS DISTRIBUTION ===")
    print(df['Status'].value_counts().to_string())

    # ── Trailer Type distribution ──────────────────────────────
    print(f"\n{separator}")
    print("=== FINAL TRAILER TYPE DISTRIBUTION ===")
    print(df['Trailer Type'].value_counts().to_string())

    # ── Shipment Type distribution ─────────────────────────────
    print(f"\n{separator}")
    print("=== FINAL SHIPMENT TYPE DISTRIBUTION ===")
    print(df['Shipment Type'].value_counts().to_string())

    # ── Financial overview ─────────────────────────────────────
    print(f"\n{separator}")
    print("=== FINANCIAL OVERVIEW ===")
    fin_cols = ['Buy Total', 'Sell Total', 'Margin Total',
                'Gross Margin', 'Margin Pct']
    available = [c for c in fin_cols if c in df.columns]
    print(df[available].describe().round(2).to_string())

    # ── Data quality flag counts ───────────────────────────────
    print(f"\n{separator}")
    print("=== DATA QUALITY FLAGS ===")
    if 'Negative Margin Flag' in df.columns:
        print(f"Negative margin rows : {df['Negative Margin Flag'].sum():,}")
    if 'Zero_Financial_Flag' in df.columns:
        print(f"Zero financial rows  : {df['Zero_Financial_Flag'].sum():,}")

    # ── Remaining nulls check ──────────────────────────────────
    print(f"\n{separator}")
    print("=== REMAINING NULL COUNTS (top 15 columns) ===")
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
    print(null_counts.head(15).to_string() if not null_counts.empty
          else "No nulls remaining in any column.")


# ── Run ──────────────────────────────────────────────────────
print_summary_statistics(df)


=== FINAL STATUS DISTRIBUTION ===
Status
Delivered           78584
Quote               16967
Canceled             2929
Complete             1242
In Transit            559
Committed              69
Out for Delivery       45
Dispatched             17

──────────────────────────────────────────────────
=== FINAL TRAILER TYPE DISTRIBUTION ===
Trailer Type
Van               56847
Flatbed           17753
LTL                9483
Reefer             3712
Unknown            3571
Power Only         3503
Hotshot            2321
Straight Truck     1968
Sprinter            753
Step Deck           167
Intermodal          146
Conestoga           141
Tanker               25
Low Boy / RGN        12
Air Freight          10

──────────────────────────────────────────────────
=== FINAL SHIPMENT TYPE DISTRIBUTION ===
Shipment Type
Truckload      57214
LTL            39987
Drayage         1664
Hot Shot         946
Unknown          530
Intermodal        33
Other             27
Air Freight       11

─────────